# 📦 **1. The Power of TENSORS**

In PyTorch, a **TENSOR** is the absolute *Main Data Structure*. Think of it as a multi-dimensional grid of numbers. While it looks like a NumPy array, it has one massive advantage: **GPU ACCELERATION**.

### 🔹 **What do Tensors represent?**
1. **INPUT DATA** (like car mileage or pixel values)
2. **MODEL PARAMETERS** (the internal weights the model learns)
3. **OUTPUTS** (the final predictions)

### 🔹 **Creating Tensors**
Let's see how we generate these containers from scratch.

In [ ]:
import torch

# Creating a tensor from a standard Python list
my_list = [[1.0, 2.0], [3.0, 4.0]]
tensor_data = torch.tensor(my_list)

#I nitializing random numbers (How models start "learning")
# This creates a 3x2 matrix of random values
random_weights = torch.randn(3, 2)

# Creating a placeholder of Zeros (Often used for biases)
biases = torch.zeros(2)

print("Manual Tensor:\n", tensor_data)
print("\nRandom Weights:\n", random_weights)

Manual Tensor:
 tensor([[1., 2.],
        [3., 4.]])

Random Weights:
 tensor([[ 0.2863, -0.6140],
        [-0.4584, -0.3219],
        [ 0.3200,  0.3104]])


# 🚀 **2. Hardware Mastery: CPU vs GPU**

By default, PyTorch lives on your **CPU**. However, training a deep model on a CPU is slow. To speed things up, we use the **GPU (CUDA)**.

### ⚠️ **The Golden Rule**
All tensors in a calculation **MUST** be on the **SAME DEVICE**. If you try to add a CPU tensor to a GPU tensor, your code will crash!

In [ ]:
# Checking for GPU availability in Colab
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Hardware detected: {device.upper()}")

# Moving a tensor to the detected device
cpu_data = torch.ones(2, 2)
print("Current location:", cpu_data.device)

# Moving to GPU
gpu_data = cpu_data.to(device)
print("New location:", gpu_data.device)

Hardware detected: CUDA
Current location: cpu
New location: cuda:0


# 🧩 **3. Shaping Your Data: VIEW vs RESHAPE**

Neural networks are extremely picky about **SHAPES**. If a layer expects 4 inputs and you give it 5, it fails. You will spend a lot of time "massaging" your tensors into the right dimensions.

### 🔹 **Common Methods**
*   **`view()`**: The fastest way to change shape. It shares memory with the original tensor.
*   **`reshape()`**: A more flexible version of view.

In [ ]:
# Create a 1D tensor of 10 numbers
x = torch.arange(10)
print("Original 1D Tensor:", x.shape)

# Change it into a 5x2 Matrix
# -1 tells PyTorch to "figure out" the missing dimension automatically
x_matrix = x.view(5, -1)
print("\nViewed as 5x2 Matrix:\n", x_matrix.shape)

# Flatten it back to 10 elements
x_flat = x_matrix.reshape(2,5)
print("\nShape after flattening:", x_flat.shape)

Original 1D Tensor: torch.Size([10])

Viewed as 5x2 Matrix:
 torch.Size([5, 2])

Shape after flattening: torch.Size([2, 5])


# 🗜️ **4. Cleaning Dimensions: SQUEEZE & FLATTEN**

Sometimes tensors come with "ghost" dimensions (dimensions with a size of 1) or grids that need to be turned into lists.

### 🔹 **The "Cleaning" Tools**
*   **`squeeze()`**: Deletes all dimensions that have a size of **1**.
*   **`flatten()`**: Collapses a multi-dimensional grid into a **single 1D list**.

In [ ]:
# Squeezing a "ghost" dimension
# Shape [1, 3] -> Shape [3]
ghost_tensor = torch.zeros(1, 3)
print("Shape before squeeze:", ghost_tensor.shape)

clean_tensor = ghost_tensor.squeeze()
print("Shape after squeeze:", clean_tensor.shape)

# Flattening a 2x2 grid for a Neural Network
grid = torch.tensor([[1, 2], [3, 4]])
flat = grid.flatten()

print("\nOriginal 2x2 Grid:\n", grid)
print("Flattened Result:", flat)

Shape before squeeze: torch.Size([1, 3])
Shape after squeeze: torch.Size([3])

Original 2x2 Grid:
 tensor([[1, 2],
        [3, 4]])
Flattened Result: tensor([1, 2, 3, 4])


# 🏡 **Part 2: Preparing Tabular Data (House Price Regression)**

Neural networks cannot read Excel files, and they cannot understand words. They only understand **Tensors**.

In this section, we will take a real-world **House Price Dataset**, clean it, fix any broken data, and transform it into PyTorch Tensors. Our goal is to predict the **Price** of a house based on features like Area, Bedrooms, and Air Conditioning.

### 🔹 **Importing via KaggleHub**
Instead of downloading files manually, we will pull the dataset directly into our Colab environment.

In [ ]:
#!pip install -q kagglehub    # un coment this if kagglehub is not installed

In [ ]:
# Downloading a dataset directly from Kaggle
import kagglehub
import pandas as pd
import glob

print("Downloading House Price dataset from Kaggle...")
path = kagglehub.dataset_download("yasserh/housing-prices-dataset")
print("Path to dataset files:", path)

# Load the CSV directly using the file path
df = pd.read_csv(f"{path}/Housing.csv")

# Let's look at the first 5 houses!
df.head()

Using Colab cache for faster access to the 'housing-prices-dataset' dataset.
Path to dataset files: /kaggle/input/housing-prices-dataset


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


# 🕵️ **Exploratory Data Analysis (EDA) & NaNs**

Real-world data is messy. Sometimes fields are left blank. These result in missing values, often called **NaN (Not a Number)**.

### ⚠️ **The Danger of NaNs**
If a Neural Network encounters a `NaN` during training, the math collapses and the model breaks. We **must** fix them before training.

In [ ]:
# Checking for Missing Values
print("--- Missing Values Before ---")
print(df.isnull().sum())

# This dataset is actually perfectly clean! But in the real world, that rarely happens.
# Let's artificially delete the 'area' values for 10 houses to simulate broken data:
import numpy as np
df.loc[0:10, 'area'] = np.nan
print("\n--- Artificially Injected NaNs in 'area' ---")
print(df.isnull().sum())

# Fixing NaNs by 'Imputing' (Filling with the Average)
# We calculate the mean of the area column, and fill the holes with that number.
average_area = df['area'].mean()
df['area'] = df['area'].fillna(average_area)

print("\n--- Missing Values After Fix ---")
print(df.isnull().sum()) # Should be 0 again!

--- Missing Values Before ---
price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64

--- Artificially Injected NaNs in 'area' ---
price                0
area                11
bedrooms             0
bathrooms            0
stories              0
mainroad             0
guestroom            0
basement             0
hotwaterheating      0
airconditioning      0
parking              0
prefarea             0
furnishingstatus     0
dtype: int64

--- Missing Values After Fix ---
price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatu

# 🔡 **3. Encoding: Turning Text into Numbers**

Our dataset has text columns like `mainroad` (yes/no) and `furnishingstatus` (furnished, semi-furnished). PyTorch only understands numbers, so we must translate these words.

### 🔹 **Dynamic Label Encoding**
Instead of typing out every single column name manually, we can write a Python script that automatically finds every column containing text (objects) and converts them into numbers using `LabelEncoder`.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Automatically finding and encoding all text columns
# 1. Find all columns where the data type is "object" (Text)
categorical_cols = df.select_dtypes(include=["object"]).columns

# 2. Loop through them and apply the Label Encoder
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

print("\nSuccess! All text has been converted to numbers.")
df.head()

Encoding column: mainroad
Encoding column: guestroom
Encoding column: basement
Encoding column: hotwaterheating
Encoding column: airconditioning
Encoding column: prefarea
Encoding column: furnishingstatus

Success! All text has been converted to numbers.


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,5068.26779,4,2,3,1,0,0,0,1,2,1,0
1,12250000,5068.26779,4,4,4,1,0,0,0,1,3,0,0
2,12250000,5068.26779,3,2,2,1,0,1,0,0,2,1,1
3,12215000,5068.26779,4,2,2,1,0,1,0,1,3,1,0
4,11410000,5068.26779,4,1,2,1,1,1,0,1,2,0,0


# 📏 **4. Feature Scaling**

Look at our data: `area` might be **8,000**, but `bedrooms` is only **3**. If we feed this directly into a Neural Network, the math will be completely overwhelmed by the massive `area` numbers and ignore the bedrooms.

We use a **StandardScaler** to squish all numerical features so they operate on a fair, level playing field (mean of 0, standard deviation of 1). We do *not* scale the target variable (`price`).

### 🔹 **Dynamic Scaling**
Just like we did with text, we will dynamically select all numerical columns to scale them automatically.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Automatically finding and scaling numerical columns
# 1. Find all numerical columns (int64 and float64)
# 2. Drop the target variable ('price') because we only scale the inputs!
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("price")

print(f"Scaling these columns: {numerical_cols.tolist()}")

# 3. Apply the Standard Scaler
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

print("\nSuccess! All numerical features are now scaled.")
# Notice how the 'area' and 'bedrooms' columns are now small decimals
df.head()

Scaling these columns: ['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus']

Success! All numerical features are now scaled.


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,4.435292e-16,1.403419,1.421812,1.378217,0.405623,-0.465315,-0.734539,-0.219265,1.472618,1.517692,1.804941,-1.406286
1,12250000,4.435292e-16,1.403419,5.405809,2.532024,0.405623,-0.465315,-0.734539,-0.219265,1.472618,2.679409,-0.554035,-1.406286
2,12250000,4.435292e-16,0.047278,1.421812,0.224410,0.405623,-0.465315,1.361397,-0.219265,-0.679063,1.517692,1.804941,-0.091662
3,12215000,4.435292e-16,1.403419,1.421812,0.224410,0.405623,-0.465315,1.361397,-0.219265,1.472618,2.679409,1.804941,-1.406286
4,11410000,4.435292e-16,1.403419,-0.570187,0.224410,0.405623,2.149083,1.361397,-0.219265,1.472618,1.517692,-0.554035,-1.406286


# ✂️ **1.5. Handling Outliers (Removing Mansions)**

Neural Networks learn by minimizing the "average" error. If our dataset contains a few $15,000,000 mansions, the model might ruin its predictions for normal, everyday houses just to avoid getting the mansion's price wrong.

These extreme cases are called **Outliers**. To help our model focus on the average homebuyer, let's remove any house priced over **$10,000,000**.

In [ ]:
print(f"Total houses before filtering: {len(df)}")

# *Example*: Filtering rows in Pandas
# We are telling Pandas to keep ONLY the rows where the price is less than or equal to 10 Million
df = df[df['price'] <= 10000000]

# Reset the index so our row numbers stay clean (0, 1, 2...)
df = df.reset_index(drop=True)

print(f"Total houses after removing outliers: {len(df)}")

# Let's see the most expensive house remaining in our dataset
max_price = df['price'].max()
print(f"Highest price remaining: ${max_price:,.2f}")

Total houses before filtering: 545
Total houses after removing outliers: 537
Highest price remaining: $9,870,000.00


# 🚚 **5. Train/Test Split & PyTorch DataLoaders**

Now that our data is 100% clean, scaled, and numeric, we need to prepare it for the Neural Network.

1. **Train/Test Split**: We split the data (80% Train, 20% Test) so we can evaluate the model on houses it has *never seen before*.
2. **Tensors**: We convert the Pandas dataframes into PyTorch Tensors.
3. **DataLoaders**: We wrap our Tensors in a DataLoader. This acts like a conveyor belt, feeding the data to the GPU in **Batches** (e.g., 32 houses at a time) instead of all at once.

In [ ]:
from sklearn.model_selection import train_test_split

# 1. TRAIN / TEST SPLIT
y = df['price'].values # The Answers
X = df.drop('price', axis=1).values # The Inputs

# Split into 80% Training, 20% Testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 2. CONVERT TO TENSORS
# For Regression, inputs and outputs MUST be float32!
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

# We use .view(-1, 1) to ensure the target is a column matrix [Batch, 1]
# instead of a flat list. This prevents huge Shape Errors later!
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)


In [ ]:

from torch.utils.data import TensorDataset, DataLoader

# 3. CREATE DATALOADERS
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

BATCH_SIZE = 32
# We shuffle the train loader so the model doesn't memorize the order of the houses!
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Data is ready! We have {len(train_loader)} batches of {BATCH_SIZE} houses ready for training.")

Data is ready! We have 14 batches of 32 houses ready for training.
